In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.1.0',
    'torchaudio>=2.1.0',
    'transformers>=4.40.0',
    'accelerate>=0.30.0',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyarrow>=16.0.0',
    'tblib>=3.0.0',
], check=True)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '*.whl'], check=True)
except Exception:
    pass

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')

from shared.secrets import load_secrets
from shared.cf_client import CFClient
from shared.workflow_kernel import WorkflowKernel
from shared.repo_router import RepoRouter

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/urdu-asr-pipelines/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = 'run_20260507_001'
SESSION_ID   = 'tpu_synth_01'
SESSION_TYPE = 'tpu_synth'
SHARD_KEY    = 'tpu'
GPU_TYPE     = 'TPU-v3-8'
VRAM_LIMIT   = 64.0

SECRETS = load_secrets(require_gemini=False)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO   = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO = repos_cfg['repos']['overflow']['repo_id']

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = GPU_TYPE,
    vram_limit_gb    = VRAM_LIMIT,
    session_max_hours = 8.5,
)
kernel.start()

# TPU / XLA setup
import torch
import torch_xla
import torch_xla.core.xla_model as xm

DEVICE = xm.xla_device()
print(f'[tpu] XLA device: {DEVICE}')

# XLA 5-bucket shape guard required for synthesis
SHAPE_BUCKETS = [2048, 4096, 8192, 16384, 32768]
print(f'[tpu] shape buckets: {SHAPE_BUCKETS}')

print(f'[session] {SESSION_ID} started — run={RUN_ID}, device={GPU_TYPE}')

In [ ]:
repo_router = RepoRouter(STAGE0_REPO, OVERFLOW_REPO)

stages_to_run = ['p5a', 'p5b', 'p5c']

for stage in stages_to_run:
    if kernel.session_expiring:
        print(f'[session] expiring — skipping {stage}')
        break
    kernel.check_session_time()

    started_at = kernel.log_stage_start(stage)
    try:
        if stage == 'p5a':
            # Synthesis — XLA 5-bucket shape guard enforced
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_5_codec_tts/p5a_synthesize.ipynb').read())
        elif stage == 'p5b':
            # Interleave synthesized audio with codec tokens
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_5_codec_tts/p5b_interleave.ipynb').read())
        elif stage == 'p5c':
            # Upload to stage45_e2e HF repo (~50GB)
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_5_codec_tts/p5c_upload.ipynb').read())
        kernel.log_stage_end(stage, started_at)
        print(f'[session] {stage} completed')
    except Exception as e:
        kernel.log_stage_end(stage, started_at, error=str(e))
        print(f'[session] {stage} failed: {e}')
        raise

kernel.stop()
print('[session] tpu_synth session complete')